In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv(r"C:\Users\Angela\Documents\My SQL folder\Weekly Checkin\Housing Analysis week 4\realtor-data.zip.csv")
df

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2226377,23009.0,sold,359900.0,4.0,2.0,0.33,353094.0,Richland,Washington,99354.0,3600.0,2022-03-25
2226378,18208.0,sold,350000.0,3.0,2.0,0.10,1062149.0,Richland,Washington,99354.0,1616.0,2022-03-25
2226379,76856.0,sold,440000.0,6.0,3.0,0.50,405677.0,Richland,Washington,99354.0,3200.0,2022-03-24
2226380,53618.0,sold,179900.0,2.0,1.0,0.09,761379.0,Richland,Washington,99354.0,933.0,2022-03-24


In [3]:
print(df.duplicated().sum())

0


In [4]:
print("shape", df.shape)

shape (2226382, 12)


In [9]:
print("\nData types:\n",df.dtypes)


Data types:
 brokered_by       float64
status             object
price             float64
bed               float64
bath              float64
acre_lot          float64
street            float64
city               object
state              object
zip_code          float64
house_size        float64
prev_sold_date     object
dtype: object


In [8]:
print("\nColumns:", df.columns.tolist())


Columns: ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']


In [10]:
print("\nMissing values:\n", df.isnull().sum())


Missing values:
 brokered_by         4533
status                 0
price               1541
bed               481317
bath              511771
acre_lot          325589
street             10866
city                1407
state                  8
zip_code             299
house_size        568484
prev_sold_date    734297
dtype: int64


In [11]:
print("\nSample:\n")


Sample:



In [12]:
df.head()

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN


##3 ANALYSIS QUESTIONS

Q1: Price by State - Which states have the highest and lowest median house prices, and which states offer the best value relative to house size?
Q2: Price per Square Foot - How does price per square foot vary across states and property sizes? Where does space cost the most?
Q3: Bedroom and Bathroom Impact - How much does each additional bedroom or bathroom add to price, and is there a point of diminishing returns?

In [13]:
# Cleaning
# Drop rows with missing price, because  price is non-negotiable for analysis
df = df.dropna(subset=['price'])
print("After dropping missing price:", len(df))

After dropping missing price: 2224841


In [14]:
# Remove clearly impossible values
# Price: remove anything below $10,000 (data errors)
# Price: remove anything above $10,000,000 (ultra-luxury outliers)
df = df[(df['price'] >= 10_000) & (df['price'] <= 10_000_000)]
print("After price filter:", len(df))

After price filter: 2201574


In [15]:
# Remove impossible bedroom/bathroom counts
df = df[(df['bed'] <= 10) & (df['bath'] <= 10)]

In [16]:
# Fill missing house_size with median per state 
df['house_size'] = df.groupby('state')['house_size'].transform(
    lambda x: x.fillna(x.median())
)

In [17]:
df.head()

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,1450.0,NaN


In [18]:
# ENGINEERING FEATURE 1 - price per square foot
df['price_per_sqft'] = (df['price'] / df['house_size']).round(2)
df['price_per_sqft'].head()

0    114.13
1     52.39
2     89.57
3     80.56
4     44.83
Name: price_per_sqft, dtype: float64

In [19]:
# ENGINEERING FEATURE 2 - price band
df['price_band'] = pd.cut(
    df['price'],
    bins = [0, 200_000, 400_000, 600_000, 1_000_000, 10_000_000],
    labels = ['Under 200k', '200k-400k', '400k-600k', '600k-1M', 'Over 1M']
)

print("\nFinal shape:", df.shape)
print("\nPrice band distribution:\n", df['price_band'].value_counts())
df.head()


Final shape: (1696600, 14)

Price band distribution:
 price_band
200k-400k     597583
400k-600k     345385
Under 200k    342257
600k-1M       255174
Over 1M       156201
Name: count, dtype: int64


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,price_per_sqft,price_band
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN,114.13,Under 200k
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN,52.39,Under 200k
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN,89.57,Under 200k
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN,80.56,Under 200k
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,1450.0,NaN,44.83,Under 200k


In [24]:
# Export cleaned data as CSV
df.to_csv('C:/Users/Angela/Documents/My SQL folder/Weekly Checkin/Housing Analysis week 4/housing_clean.csv', index=False)

C:/Users/Angela/Documents/My SQL folder/Weekly Checkin/Housing Analysis week 4/housing_clean.csv 1696600 rows


In [25]:
check = pd.read_csv('C:/Users/Angela/Documents/My SQL folder/Weekly Checkin/Housing Analysis week 4/housing_clean.csv')
print(check.shape)   # should match your df shape
print(check.head())

(1696600, 14)
   brokered_by    status     price  bed  bath  acre_lot     street  \
0     103378.0  for_sale  105000.0  3.0   2.0      0.12  1962661.0   
1      52707.0  for_sale   80000.0  4.0   2.0      0.08  1902874.0   
2     103379.0  for_sale   67000.0  2.0   1.0      0.15  1404990.0   
3      31239.0  for_sale  145000.0  4.0   2.0      0.10  1947675.0   
4      34632.0  for_sale   65000.0  6.0   2.0      0.05   331151.0   

         city        state  zip_code  house_size prev_sold_date  \
0    Adjuntas  Puerto Rico     601.0       920.0            NaN   
1    Adjuntas  Puerto Rico     601.0      1527.0            NaN   
2  Juana Diaz  Puerto Rico     795.0       748.0            NaN   
3       Ponce  Puerto Rico     731.0      1800.0            NaN   
4    Mayaguez  Puerto Rico     680.0      1450.0            NaN   

   price_per_sqft  price_band  
0          114.13  Under 200k  
1           52.39  Under 200k  
2           89.57  Under 200k  
3           80.56  Under 200k  
4 

In [26]:
print(df['price'].dtype)
print(df['price'].head(10))
print(df['price'].unique()[:20])  # see what unique values exist

float64
0    105000.0
1     80000.0
2     67000.0
3    145000.0
4     65000.0
5    179000.0
6     50000.0
7     71600.0
8    100000.0
9    300000.0
Name: price, dtype: float64
[105000.  80000.  67000. 145000.  65000. 179000.  50000.  71600. 100000.
 300000.  89000. 150000. 155000.  79000. 649000. 120000. 235000. 575000.
 140000. 165000.]


In [27]:
print(df.iloc[297]['price'])        # row 298 (0-indexed = 297)
print(df.iloc[295:300]['price'])    # see surrounding rows too

76900.0
409    265000.0
410     99000.0
411     76900.0
412     95000.0
413    245000.0
Name: price, dtype: float64


In [28]:
# Check for any non-numeric or problematic values
print(df[df['price'].astype(str).str.contains('[^0-9.]', regex=True)]['price'].head(10))

Series([], Name: price, dtype: float64)


In [29]:
df.iloc[297]

brokered_by           52270.0
status               for_sale
price                 76900.0
bed                       3.0
bath                      2.0
acre_lot                  NaN
street              1873083.0
city                Canovanas
state             Puerto Rico
zip_code                729.0
house_size             1200.0
prev_sold_date     2020-02-28
price_per_sqft          64.08
price_band         Under 200k
Name: 411, dtype: object

In [30]:
print("\nColumns:", df.columns.tolist())


Columns: ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'street', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date', 'price_per_sqft', 'price_band']
